In [ ]:
import numpy as np

In [ ]:
# PARAMETRI SPERIMENTALI
# ----------------------------------------------------------------------------

SIGMA_M = np.sqrt((0.1/np.sqrt(12))**2 + (0.3)**2)     # g, risoluzione bilancia + linearity
SIGMA_T = np.sqrt((0.2/np.sqrt(12))**2 + (1)**2)  # °C, incertezza termometro + accuracy del termometro

print(f"SIGMA_M = {SIGMA_M:.3f} g")
print(f"SIGMA_T = {SIGMA_T:.3f} °C")

ME_ASSUNTA = 25.0       # g, valore usato nelle altre misure
SIGMA_ME_ASSUNTA = 5.0  # g

In [ ]:
# m1, T1: acqua calda nel thermos; m2, T2: acqua fredda versata
m1 = [102.1, 99.2]  # (misura 2 scartata: Me < 0, ghiaccio nell'acqua fredda)
T1 = [47.6, 50.8]   # °C
m2 = [100.8, 100.3] # g
T2 = [6.0, 4.8]     # °C
Tf = [28.0, 26.4]   # °C

Me = []
sigma_stats = []

In [ ]:
for i in range(len(m1)):
    Me.append(m2[i] * (T2[i] - Tf[i]) / (Tf[i] - T1[i]) - m1[i])
    sigma_stats.append(np.sqrt(((T2[i] - Tf[i])/(Tf[i] - T1[i]) * SIGMA_M)**2 +
                       (SIGMA_M)**2 +
                       (m2[i]/(Tf[i] - T1[i]) * SIGMA_T)**2 +
                       (m2[i] * (T2[i] - Tf[i])/(Tf[i] - T1[i])**2 * SIGMA_T)**2 +
                       (m2[i] * (T1[i] - T2[i])/(Tf[i] - T1[i])**2 * SIGMA_T)**2))

for i in range(len(m1)):
    print(f"Misura {i+1}: Me = {Me[i]:.1f} ± {sigma_stats[i]:.1f} g | scarto da Me assunta: {abs(Me[i] - ME_ASSUNTA)/np.sqrt(sigma_stats[i]**2 + SIGMA_ME_ASSUNTA**2):.1f} sigma")

In [ ]:
# usando la seconda misura calcoliamo la massa di ghiaccio all'interno del thermos
# Me "vera" dalla misura 1 o dal valore assunto, dati sperimentali dalla misura 2
r"""
\begin{equation}
    \delta = \frac{(m_1 + M_e)(T_1 - T_f) - m_2 (T_f - T_2)}{\lambda + T_2}
\end{equation}

\begin{equation}
\begin{aligned}
\sigma_\delta = & \frac{1}{\lambda + T_2}\left((T_1 - T_f)^2\left(\sigma_m^2 + \sigma_{M_e}^2\right) + (T_f - T_2)^2\sigma_m^2 \right.\\
& \left. + \left[(m_1 + M_e)^2 + (m_1 + M_e + m_2)^2 + (m_2 - \delta)^2\right]\sigma_T^2\right)^\frac{1}{2}   
\end{aligned}
\end{equation}
"""

lamda = 79.71
D = lamda + T2[1]

In [ ]:
# riferimenti per Me "vera": misura 1 e valore assunto
riferimenti = {"misura 1": (Me[0], sigma_stats[0]),
               "Me assunta": (ME_ASSUNTA, SIGMA_ME_ASSUNTA)}

In [ ]:
for nome, (ME_VERA, SIGMA_ME_VERA) in riferimenti.items():
    delta = ((m1[1] + ME_VERA) * (T1[1] - Tf[1]) - m2[1] * (Tf[1] - T2[1])) / D
    sigma_delta = np.sqrt(((T1[1] - Tf[1])/D)**2 * (SIGMA_M**2 + SIGMA_ME_VERA**2) +   # m_1, M_e
                          ((Tf[1] - T2[1])/D * SIGMA_M)**2 +                            # m_2
                          ((m1[1] + ME_VERA)/D * SIGMA_T)**2 +                          # T_1
                          ((m1[1] + ME_VERA + m2[1])/D * SIGMA_T)**2 +                  # T_f
                          ((m2[1] - delta)/D * SIGMA_T)**2)                             # T_2

    print(f"Delta (Me da {nome} = {ME_VERA:.1f} ± {SIGMA_ME_VERA:.1f} g) = {delta:.1f} ± {sigma_delta:.1f} g | scarto da 0: {abs(delta)/sigma_delta:.1f} sigma")